### Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col,trim
from pyspark.sql.types import StringType,DataType

### Read Bronze table

In [0]:
df = spark.table("workspace.bronze.erp_cust_az12")
print(f"Bronze row count: {df.count()}")
df.display()

### Silver Transformations

### TRIMMING

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df = df.withColumn(field.name,trim(col(field.name)))

### customer ID cleanup

In [0]:
df = df.withColumn(
    "cid",
      F.when(col("cid").startswith("NAS"),
      F.substring(col("cid"),4,F.length(col("cid"))))
       .otherwise(col("cid"))
)

### Birthdate validation

In [0]:
Inavalid_birthdate= df.filter(F.col("bdate") > F.current_date()).count()
print(f"Invalid future date founds: {Inavalid_birthdate} - setting to Null")
df = df.withColumn(
    "bdate",
    F.when(col("bdate") > F.current_date(),None)
    .otherwise(col("bdate"))
)

### Gender Normalization

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.upper(col("GEN")).isin("F","FEMALE"),"Female")
     .when(F.upper(col("GEN")).isin("M","MALE"),"Male")
     .otherwise("n/a")
)

### Renaming columns

In [0]:
RENAME_MAP = {
    "cid": "customer_number",
    "bdate": "birth_date",
    "gen": "gender"}
for old_name,new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name,new_name)

### Sanity checks of dataframe

In [0]:
print("Sample Data :")
display(df.limit(10))
print("\n Gender Distibution:")
df.groupBy("gender").count().display()


### Writing Silver Table

In [0]:
%sql
drop table workspace.silver.erp_customers

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customers")
print(f"Written {df.count()} rows to workspace.silver.erp_customers")

### Sanity checks of silver table

In [0]:
%sql
select * from workspace.silver.erp_customers